# Reduced States and Mixed Gaussian States

After a matchgate circuit, the state of a subset of the wires is obtained by tracing out the
other wires. This reduced state is generally mixed, but it remains a fermionic Gaussian state,
fully described by its Majorana covariance matrix
$\Lambda_{\mu\nu} = i\,\mathrm{Tr}(\rho\, c_\mu c_\nu)$ restricted to the kept wires. MatchCake
can therefore

- read the covariance matrix, the Von Neumann entropy, the purity and the density matrix of a
  subsystem with measurements in a `QNode`;
- start another circuit from a mixed Gaussian state with `mc.GaussianStatePrep`, for example
  from the reduced state of a previous circuit;
- differentiate through the whole chain.

The theory is detailed in
[Tracing out wires and mixed Gaussian states](https://github.com/MatchCake/MatchCake/blob/main/docs/partial_trace_theory.md).

In [1]:
import numpy as np
import pennylane as qml
import torch

import matchcake as mc
from matchcake.operations import CompRxRx, CompRyRy, CompRzRz, SptmCompRxRx, SptmCompRyRy, SptmCompRzRz
from matchcake.utils import CovarianceMatrixToolbox

## Matchgate circuit

We use a brick-wall circuit of matchgates on 6 qubits, applied to the basis state
$|101010\rangle$. Every gate is also understood by `default.qubit`, which we use as a reference.

In [2]:
n = 6
wires = list(range(n))
basis_state = np.array([1, 0, 1, 0, 1, 0])
rng = np.random.default_rng(0)
angles = rng.uniform(0, np.pi, size=(2, n - 1, 3, 2))


def brick_wall(layer_angles, circuit_wires):
    # layer_angles: (n_layers, len(circuit_wires) - 1, 3, 2)
    for layer in layer_angles:
        for position, gate_angles in enumerate(layer):
            pair = [circuit_wires[position], circuit_wires[position + 1]]
            CompRxRx(gate_angles[0], wires=pair)
            CompRyRy(gate_angles[1], wires=pair)
            CompRzRz(gate_angles[2], wires=pair)


def first_circuit(measurement):
    qml.BasisState(basis_state, wires=wires)
    brick_wall(angles, wires)
    return measurement()


def run_first_circuit(device_name, measurement):
    return qml.QNode(first_circuit, qml.device(device_name, wires=n))(measurement)

## Reading a subsystem

The wires given to a measurement are the wires that are kept; the others are traced out.
`qml.vn_entropy` and `qml.purity` are the usual PennyLane measurements. With a pure state on
the whole register, the entropy of a subsystem is the entanglement entropy between the
subsystem and the rest.

In [3]:
kept_wires = [0, 1, 2, 3]


def subsystem_measurements():
    return qml.vn_entropy(wires=kept_wires), qml.purity(wires=kept_wires)


entropy, purity = run_first_circuit("nif.qubit", subsystem_measurements)
expected_entropy, expected_purity = run_first_circuit("default.qubit", subsystem_measurements)
print(f"entropy = {float(entropy):.6f}  (default.qubit: {float(expected_entropy):.6f})")
print(f"purity  = {float(purity):.6f}  (default.qubit: {float(expected_purity):.6f})")
np.testing.assert_allclose(entropy, expected_entropy, atol=1e-8)
np.testing.assert_allclose(purity, expected_purity, atol=1e-8)

entropy = 0.878858  (default.qubit: 0.878858)
purity  = 0.468633  (default.qubit: 0.468633)


The cost does not depend on the size of the Hilbert space: both quantities are computed from
the $2k \times 2k$ covariance matrix of the $k$ kept wires. The entanglement entropy of every
cut of a larger system is then cheap.

In [4]:
n_large = 40
large_wires = list(range(n_large))
large_angles = rng.uniform(0, np.pi, size=(3, n_large - 1, 3, 2))
cuts = [5, 10, 20, 30, 35]


@qml.qnode(mc.NIFDevice(wires=n_large))
def large_circuit():
    qml.BasisState(np.arange(n_large) % 2, wires=large_wires)
    brick_wall(large_angles, large_wires)
    return [qml.vn_entropy(wires=large_wires[:cut]) for cut in cuts]


for cut, cut_entropy in zip(cuts, large_circuit()):
    print(f"entropy of wires 0..{cut - 1:<2} = {float(cut_entropy):.4f}")

entropy of wires 0..4  = 1.4169
entropy of wires 0..9  = 1.3000
entropy of wires 0..19 = 0.7734
entropy of wires 0..29 = 0.8444
entropy of wires 0..34 = 1.0176


## The reduced covariance matrix

`mc.covariance_matrix(wires)` returns the covariance matrix $\Lambda_A$ of the kept wires. It
describes the reduced state completely. Its normal-mode polarizations $|\lambda_j| \le 1$ tell
how mixed the state is: a mode with $|\lambda_j| = 1$ is pure, and at most as many modes are
mixed as there are traced-out wires.

For a small number of wires, `qml.density_matrix(wires)` returns the dense reduced density
matrix, which is also what `CovarianceMatrixToolbox.to_density_matrix` computes from $\Lambda_A$.

In [5]:
reduced_covariance = np.asarray(run_first_circuit("nif.qubit", lambda: mc.covariance_matrix(wires=kept_wires)))
print(f"reduced covariance shape = {reduced_covariance.shape}")
print(f"mode polarizations       = {np.round(CovarianceMatrixToolbox.mode_polarizations(reduced_covariance), 4)}")

expected_density = run_first_circuit("default.qubit", lambda: qml.density_matrix(wires=kept_wires))
np.testing.assert_allclose(CovarianceMatrixToolbox.to_density_matrix(reduced_covariance), expected_density, atol=1e-8)
np.testing.assert_allclose(
    run_first_circuit("nif.qubit", lambda: qml.density_matrix(wires=kept_wires)), expected_density, atol=1e-8
)
print("The reduced density matrix agrees with state-vector simulation.")

reduced covariance shape = (8, 8)
mode polarizations       = [0.2033 0.8945 1.     1.    ]
The reduced density matrix agrees with state-vector simulation.


## Continuing on the reduced state

`mc.GaussianStatePrep(covariance_matrix, wires)` prepares the Gaussian state with the given
covariance matrix. Queued as the first operation of a circuit, it starts a 4-qubit simulation
from the mixed reduced state of the 6-qubit circuit. Expectation values, probabilities and
samples are then computed as usual.

The reference applies the same follow-up gates to the kept wires of the full 6-qubit system.

In [6]:
n_kept = len(kept_wires)
follow_up_angles = rng.uniform(0, np.pi, size=(1, n_kept - 1, 3, 2))
observable = qml.X(0) @ qml.Z(1) @ qml.X(2) + 0.5 * qml.Z(0) @ qml.Z(3)


@qml.qnode(mc.NIFDevice(wires=n_kept))
def follow_up_circuit(covariance_matrix):
    mc.GaussianStatePrep(covariance_matrix, wires=range(n_kept))
    brick_wall(follow_up_angles, list(range(n_kept)))
    return qml.expval(observable), qml.probs(wires=[0, 1])


@qml.qnode(qml.device("default.qubit", wires=n))
def full_reference_circuit():
    qml.BasisState(basis_state, wires=wires)
    brick_wall(angles, wires)
    brick_wall(follow_up_angles, kept_wires)
    return qml.expval(observable), qml.probs(wires=[0, 1])


expval, probs = follow_up_circuit(reduced_covariance)
expected_expval, expected_probs = full_reference_circuit()
print(f"expval = {float(expval):.6f}  (default.qubit: {float(expected_expval):.6f})")
print(f"probs  = {np.round(np.asarray(probs), 4)}")
np.testing.assert_allclose(expval, expected_expval, atol=1e-8)
np.testing.assert_allclose(probs, expected_probs, atol=1e-8)

expval = 0.374603  (default.qubit: 0.374603)
probs  = [0.391  0.2631 0.0528 0.2931]


With shots, the device samples the mixed state directly. The follow-up gates are written here
with their single-particle transition matrix counterparts (`SptmCompRxRx`, ...), which act
directly on the covariance matrix.

In [7]:
_ = torch.manual_seed(0)


@qml.set_shots(20_000)
@qml.qnode(mc.NIFDevice(wires=n_kept))
def sampled_follow_up_circuit(covariance_matrix):
    mc.GaussianStatePrep(covariance_matrix, wires=range(n_kept))
    for position, gate_angles in enumerate(follow_up_angles[0]):
        SptmCompRxRx(gate_angles[0], wires=[position, position + 1])
        SptmCompRyRy(gate_angles[1], wires=[position, position + 1])
        SptmCompRzRz(gate_angles[2], wires=[position, position + 1])
    return qml.probs(wires=[0, 1])


estimated_probs = sampled_follow_up_circuit(reduced_covariance)
print(f"estimated probs = {np.round(np.asarray(estimated_probs), 4)}")
np.testing.assert_allclose(estimated_probs, expected_probs, atol=2e-2)

estimated probs = [0.39   0.2644 0.0512 0.2944]


Outside of a `QNode`, the device gives the same reduced state as a ready-to-queue state
preparation with `reduced_state` (wires to keep) or `partial_trace` (wires to remove).

In [8]:
device = mc.NIFDevice(wires=n)
with qml.queuing.AnnotatedQueue() as queue:
    qml.BasisState(basis_state, wires=wires)
    brick_wall(angles, wires)
device.apply(queue.queue)

reduced_state = device.partial_trace([4, 5])
print(f"{reduced_state.name} on wires {reduced_state.wires.tolist()}")
np.testing.assert_allclose(reduced_state.covariance_matrix, reduced_covariance, atol=1e-8)

GaussianStatePrep on wires [0, 1, 2, 3]


Inside a `QNode`, the measurement `mc.reduced_state(wires)` returns the same state preparation.
It is queued with `qml.apply` at the start of the next circuit. Unlike the covariance matrix, it
also carries the displacement vector of a product state with superposed qubits and the gate
barriers of the last section.

In [9]:
@qml.qnode(mc.NIFDevice(wires=n))
def first_circuit_state():
    qml.BasisState(basis_state, wires=wires)
    brick_wall(angles, wires)
    return mc.reduced_state(wires=kept_wires)


@qml.qnode(mc.NIFDevice(wires=n_kept))
def follow_up_from_state(state):
    qml.apply(state)
    brick_wall(follow_up_angles, list(range(n_kept)))
    return qml.expval(observable)


state_expval = follow_up_from_state(first_circuit_state())
print(f"expval = {float(state_expval):.6f}  (default.qubit: {float(expected_expval):.6f})")
np.testing.assert_allclose(state_expval, expected_expval, atol=1e-8)

expval = 0.374603  (default.qubit: 0.374603)


## Chaining circuits with generators

The same chain works with `execute_generator`, which consumes the operations one at a time
instead of a queued circuit. The trace-out is a method of the device and not an operation, so it
cannot be yielded in the middle of a generator. Each segment of the circuit is one call to
`execute_generator`, and the reduced state returned by `partial_trace` is the first operation
of the next segment.

In [10]:
def brick_wall_generator(layer_angles, circuit_wires):
    for layer in layer_angles:
        for position, gate_angles in enumerate(layer):
            pair = [circuit_wires[position], circuit_wires[position + 1]]
            yield CompRxRx(gate_angles[0], wires=pair)
            yield CompRyRy(gate_angles[1], wires=pair)
            yield CompRzRz(gate_angles[2], wires=pair)


def first_segment():
    yield qml.BasisState(basis_state, wires=wires)
    yield from brick_wall_generator(angles, wires)


def follow_up_segment(state_prep, circuit_wires):
    yield state_prep
    yield from brick_wall_generator(follow_up_angles, circuit_wires)


large_device = mc.NIFDevice(wires=n)
large_device.execute_generator(first_segment())
reduced_state = large_device.partial_trace([4, 5])

small_device = mc.NIFDevice(wires=n_kept)
generator_expval = small_device.execute_generator(
    follow_up_segment(reduced_state, list(range(n_kept))), observable=observable, output_type="expval"
)
print(f"expval = {float(generator_expval):.6f}  (default.qubit: {float(expected_expval):.6f})")
np.testing.assert_allclose(generator_expval, expected_expval, atol=1e-8)

expval = 0.374603  (default.qubit: 0.374603)


### Reusing the same device

A new device is not required after a trace-out. The same device can continue from the reduced
state with two arguments:

- `reduced_wires` labels the state preparation with the kept wires of the device. The state is
  then extended to the whole register, with the traced-out wires back in $|0\rangle$.
- `reset=True` clears the operations of the previous segment. Without it, the device keeps its
  previous state and, by default, does not apply the new operations.

The device keeps its number of wires, so the following segments are only cheaper on a smaller
device.

In [11]:
reduced_state = large_device.partial_trace([4, 5], reduced_wires=kept_wires)
same_device_expval = large_device.execute_generator(
    follow_up_segment(reduced_state, kept_wires), observable=observable, output_type="expval", reset=True
)
print(f"expval = {float(same_device_expval):.6f}  (default.qubit: {float(expected_expval):.6f})")
np.testing.assert_allclose(same_device_expval, expected_expval, atol=1e-8)

expval = 0.374603  (default.qubit: 0.374603)


## Other mixed Gaussian states

A `GaussianStatePrep` does not have to come from a partial trace. `from_polarizations` builds
the product of single-mode mixtures
$\tfrac{1+\lambda_j}{2}|0\rangle\langle 0| + \tfrac{1-\lambda_j}{2}|1\rangle\langle 1|$, such as
the thermal state of $H = \sum_j \epsilon_j Z_j / 2$ at inverse temperature $\beta$, for which
$\lambda_j = -\tanh(\beta \epsilon_j / 2)$.

In [12]:
beta = 1.5
energies = np.array([0.5, 1.0, 1.5, 2.0])
polarizations = -np.tanh(beta * energies / 2)


@qml.qnode(mc.NIFDevice(wires=n_kept))
def thermal_circuit():
    mc.GaussianStatePrep.from_polarizations(polarizations, wires=range(n_kept))
    brick_wall(follow_up_angles, list(range(n_kept)))
    return qml.expval(observable), qml.vn_entropy(wires=range(n_kept))


thermal_expval, thermal_entropy = thermal_circuit()
print(f"expval  = {float(thermal_expval):.6f}")
print(f"entropy = {float(thermal_entropy):.6f}  (unchanged by the unitary circuit)")

expval  = 0.139542
entropy = 1.608146  (unchanged by the unitary circuit)


## Gate barriers

A matrix cannot remember which wires were removed. When a removed wire lies between two kept
wires, the Jordan-Wigner strings of the kept Majorana operators run through it, and the
covariance matrix restricted to the kept wires describes the reduced state of the
*fermionic modes*, which is not the reduced state of the qubits. `mc.covariance_matrix` and the
entropy measurements therefore raise an error for such wires.

`mc.reduced_state`, `reduced_state` and `partial_trace` accept any kept wires. The state they
return carries a *gate barrier* at the place of the removed wire. Gates on one side of the
barrier, probabilities, samples and the observables with an even number of $X$ and $Y$ on the
right of the barrier are exact. The device raises a `DeviceError` for the other gates and
observables, whose result would not be the one of the reduced state of the qubits.
`gate_barrier=False` returns the state without barrier, with a warning.

In [13]:
try:
    run_first_circuit("nif.qubit", lambda: qml.vn_entropy(wires=[0, 2]))
except ValueError as error:
    print(f"ValueError: {error}")

gap_wires = [0, 2, 3]
gap_state = run_first_circuit("nif.qubit", lambda: mc.reduced_state(wires=gap_wires))
print(f"\nbarriers = {gap_state.barriers}")


@qml.qnode(mc.NIFDevice(wires=len(gap_wires)))
def gap_circuit(gap_observable):
    qml.apply(gap_state)
    CompRxRx(follow_up_angles[0, 0, 0], wires=[1, 2])
    return qml.expval(gap_observable)


@qml.qnode(qml.device("default.qubit", wires=n))
def gap_reference_circuit():
    qml.BasisState(basis_state, wires=wires)
    brick_wall(angles, wires)
    CompRxRx(follow_up_angles[0, 0, 0], wires=[2, 3])
    return qml.expval(qml.Z(0) @ qml.X(2) @ qml.Y(3))


gap_expval = gap_circuit(qml.Z(0) @ qml.X(1) @ qml.Y(2))
print(f"expval = {float(gap_expval):.6f}  (default.qubit: {float(gap_reference_circuit()):.6f})")
np.testing.assert_allclose(gap_expval, gap_reference_circuit(), atol=1e-8)

try:
    gap_circuit(qml.X(0) @ qml.X(1))
except qml.exceptions.DeviceError as error:
    print(f"\nDeviceError: {error}")

ValueError: The kept wires (device positions [0, 2]) are not a block of neighbouring wires listed in the device wire order without gate barrier on nif.qubit: a removed wire or a gate barrier lies between them. The Jordan-Wigner strings of their Majorana operators then run through removed wires, so the covariance matrix restricted to them describes the reduced state of the fermionic modes, which differs from the reduced state of the qubits and from what a qubit simulator returns. A matrix cannot carry the gate barriers that make this safe: use reduced_state, partial_trace or matchcake.reduced_state to get the reduced state with its barriers, or pass gate_barrier=False to get the covariance matrix of the fermionic reduced state.

barriers = (1,)


expval = -0.387392  (default.qubit: -0.387392)

DeviceError: The Pauli word X(0) @ X(1) of the observable X(0) @ X(1) has an odd number of X and Y on the right of the gate barrier before the device position 1 on nif.qubit. The initial state is a reduced state: wires were traced out at this barrier, and the expectation value of this word in it is not the one of the reduced state of the qubits.


Expectation values and probabilities on non-contiguous wires are not concerned: they are
computed from the covariance matrix of the full register.

## Gradients

All of the above is differentiable with torch. Here the parameters of the first circuit are
optimized through the reduced covariance matrix and the second circuit, and the entropy of the
cut is differentiated as well.

In [14]:
torch_angles = torch.tensor(angles, requires_grad=True)
torch_follow_up_angles = torch.tensor(follow_up_angles)


@qml.qnode(mc.NIFDevice(wires=n), interface="torch")
def torch_first_circuit(layer_angles):
    qml.BasisState(basis_state, wires=wires)
    brick_wall(layer_angles, wires)
    return mc.covariance_matrix(wires=kept_wires), qml.vn_entropy(wires=kept_wires)


@qml.qnode(mc.NIFDevice(wires=n_kept), interface="torch")
def torch_follow_up_circuit(covariance_matrix):
    mc.GaussianStatePrep(covariance_matrix, wires=range(n_kept))
    brick_wall(torch_follow_up_angles, list(range(n_kept)))
    return qml.expval(observable)


covariance_matrix, cut_entropy = torch_first_circuit(torch_angles)
loss = torch_follow_up_circuit(covariance_matrix) + 0.1 * cut_entropy
loss.backward()
print(f"loss               = {loss.item():.6f}")
print(f"gradient shape     = {tuple(torch_angles.grad.shape)}")
print(f"gradient magnitude = {torch_angles.grad.norm().item():.6f}")

loss               = 0.462489
gradient shape     = (2, 5, 3, 2)


gradient magnitude = 0.853773


## References

1. S. Bravyi, *Lagrangian representation for fermionic linear optics*, Quantum Inf. Comput. 5,
   216 (2005), [arXiv:quant-ph/0404180](https://arxiv.org/abs/quant-ph/0404180).
2. I. Peschel, *Calculation of reduced density matrices from correlation functions*,
   J. Phys. A 36, L205 (2003), [arXiv:cond-mat/0212631](https://arxiv.org/abs/cond-mat/0212631).